# 装饰器：从设计模式到 Python 的 @

装饰器有两个身份：

- **装饰器设计模式（Decorator Pattern）**：GoF 23 种设计模式中的结构型模式，核心思想是「用包装（组合）代替继承，在不改动原对象的前提下动态叠加职责」。
- **Python 装饰器**：Python 用 `@` 语法为这个思想提供的语言级支持，用一个可调用对象包装另一个可调用对象。

本篇先手动实现设计模式版本，再看 Python 如何用「函数是一等对象」把同样的思想压缩成一行 `@`。

## 一、装饰器设计模式：包装代替继承

问题：一杯咖啡 10 元，有人要加奶（+3 元），有人加糖（+1.5 元），有人双份奶加糖……如果为每种组合都建一个子类，n 种配料对应 2ⁿ 个类，而且继承关系写死在代码里，无法在运行期按需组合。

装饰器模式的解法：

- 定义统一的抽象组件接口（`Coffee`）。
- 被装饰者（`SimpleCoffee`）实现接口。
- 装饰者（`MilkDecorator`、`SugarDecorator`）**也实现同一接口**，内部**持有**一个组件对象（组合），调用时先委托给内部对象，再叠加自己的职责。

注意一个关键细节：装饰者继承接口只是为了「类型兼容」（让装饰者还能继续被装饰），行为扩展靠的是**组合**而不是继承。

In [1]:
from abc import ABC, abstractmethod


class Coffee(ABC):
    @abstractmethod
    def cost(self) -> float: ...

    @abstractmethod
    def description(self) -> str: ...


class SimpleCoffee(Coffee):
    def cost(self) -> float:
        return 10.0

    def description(self) -> str:
        return "咖啡"


class MilkDecorator(Coffee):
    def __init__(self, coffee: Coffee):
        self.coffee = coffee  # 组合：持有被装饰者

    def cost(self) -> float:
        return self.coffee.cost() + 3.0  # 先委托，再加自己的职责

    def description(self) -> str:
        return f"{self.coffee.description()} + 牛奶"


class SugarDecorator(Coffee):
    def __init__(self, coffee: Coffee):
        self.coffee = coffee

    def cost(self) -> float:
        return self.coffee.cost() + 1.5

    def description(self) -> str:
        return f"{self.coffee.description()} + 糖"


order = SugarDecorator(MilkDecorator(SimpleCoffee()))
print(order.description())
print(f"总价：{order.cost()} 元")

咖啡 + 牛奶 + 糖
总价：14.5 元


`MilkDecorator` 不关心里面包的是纯咖啡还是已经加过糖的咖啡——只要实现了 `Coffee` 接口就能继续包一层。想要「加奶加糖」就嵌套两层，想要新配料就再写一个装饰者类，原有代码一行都不用改。这就是「对扩展开放、对修改关闭」。

## 二、前置知识：Python 的函数是一等对象

设计模式版实现比较繁琐：每加一种职责都要定义一个类。Python 里函数本身就是对象，可以赋值给变量、当作参数传递、作为返回值、存进容器——这意味着「包装并增强一个函数」不需要类的仪式，直接用函数就行。

In [2]:
def add(a, b):
    return a + b


def sub(a, b):
    return a - b


operations = {"加": add, "减": sub}  # 函数可以放进字典


def apply(func, a, b):  # 函数可以作为参数
    return func(a, b)


def make_greeter(word):  # 函数可以作为返回值
    def greet(name):
        return f"{word}, {name}"
    return greet


print(apply(add, 1, 2), apply(sub, 10, 4))
print(operations["减"](10, 3))
hello = make_greeter("你好")
print(hello("小明"))

3 6
7
你好, 小明


## 三、手动装饰：不使用 @

把「包装函数、返回增强版新函数」写成通用形式：`loud` 接收一个函数，返回一个 `wrapper`。然后手动把原函数的名字指向包装结果。

这一步就是装饰器模式的函数版：**包一层、保持可调用、在调用前后插入逻辑**。

In [3]:
def greet(name):
    return f"hello, {name}!"


def loud(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()

    return wrapper


greet = loud(greet)  # 手动包装：名字不变，行为增强
print(greet("xiaoming"))

HELLO, XIAOMING!


## 四、@ 语法糖：装饰发生在「定义时」

`@loud` 只是省去手动赋值这一步：

```python
@loud
def greet(name): ...

# 等价于
def greet(name): ...
greet = loud(greet)
```

两个常见误解要纠正：装饰**不是**每次调用时都发生，而是 `def` 语句执行（模块导入或函数定义）时**只执行一次**；装饰之后，`greet` 这个名字指向的已经是 `wrapper`，不再是原函数。

下面的单元格用一条 print 观察装饰发生的时机。

In [4]:
def loud(func):
    print(f"正在装饰：{func.__name__}")  # 用来观察装饰发生的时机

    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()

    return wrapper


@loud
def greet(name):
    return f"hello, {name}!"


print("函数定义完成，还没有调用过 greet")
print(greet("xiaoming"))
print(greet("hanmeimei"))

正在装饰：greet
函数定义完成，还没有调用过 greet
HELLO, XIAOMING!
HELLO, HANMEIMEI!


## 五、通用包装：*args 与 **kwargs

`wrapper(*args, **kwargs)` 让装饰器不关心被装饰函数的签名——参数原样转交，返回值原样返回。配合前面的思路，可以写一个真正实用的计时装饰器。

不过下面的写法有个副作用，运行后能看到。

In [5]:
import time


def timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} 耗时 {time.perf_counter() - start:.6f} 秒")
        return result

    return wrapper


@timer
def slow_sum(n):
    """求 0..n-1 的和"""
    return sum(range(n))


print(slow_sum(1_000_000))
print(slow_sum.__name__)  # 不是 slow_sum 了，而是 wrapper
print(slow_sum.__doc__)   # 文档字符串也丢了

slow_sum 耗时 0.011525 秒
499999500000
wrapper
None


## 六、别弄丢原函数的名字：functools.wraps

用 `wrapper` 替换原函数后，`__name__`、`__doc__` 等元信息变成 wrapper 自己的，调试、文档工具和序列化都会受影响。`@functools.wraps(func)` 负责把这些元信息拷贝到 wrapper 上，写装饰器时应当养成习惯。

In [6]:
import functools
import time


def timer(func):
    @functools.wraps(func)  # 把 func 的元信息拷贝给 wrapper
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} 耗时 {time.perf_counter() - start:.6f} 秒")
        return result

    return wrapper


@timer
def slow_sum(n):
    """求 0..n-1 的和"""
    return sum(range(n))


print(slow_sum(1_000_000))
print(slow_sum.__name__, "--", slow_sum.__doc__)

slow_sum 耗时 0.009793 秒
499999500000
slow_sum -- 求 0..n-1 的和


## 七、带参数的装饰器：再包一层

如果装饰器本身需要配置（比如「重复几次」），就得再包一层：`repeat(times)` 返回真正的装饰器 `decorator`，`decorator` 再返回 `wrapper`。`@repeat(times=3)` 会先求值 `repeat(times=3)` 拿到装饰器，再用它装饰下面的函数。

现在共有三层函数：最外层收**装饰器参数**，中间层收**被装饰函数**，最里层收**调用参数**。

In [7]:
import functools


def repeat(times):
    def decorator(func):                       # 第二层：收被装饰函数
        @functools.wraps(func)
        def wrapper(*args, **kwargs):          # 第三层：收调用参数
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result

        return wrapper

    return decorator


@repeat(times=3)
def say_hi(name):
    print(f"你好，{name}")


say_hi("小明")

你好，小明
你好，小明
你好，小明


## 八、多个装饰器叠加：靠近 def 的先包

装饰顺序从下往上，调用顺序从外往里：`@bold` 在 `@italic` 之上，等价于 `bold(italic(hello))`——`italic` 先包住原函数，`bold` 再包住 `italic` 的结果。

In [8]:
import functools


def bold(func):
    @functools.wraps(func)
    def wrapper():
        return f"<b>{func()}</b>"

    return wrapper


def italic(func):
    @functools.wraps(func)
    def wrapper():
        return f"<i>{func()}</i>"

    return wrapper


@bold
@italic
def hello():
    return "hello"


print(hello())  # italic 先执行，bold 后执行

<b><i>hello</i></b>


## 九、用类实现装饰器：回到设计模式

绕了一圈又回到设计模式：`@` 后面可以是任何「接收可调用对象、返回可调用对象」的东西，类实例也能胜任——实现 `__call__` 让实例像函数一样可被调用（详见 `3-__call__` 那篇笔记）。

类装饰器的优势是**能保存状态**：下面的 `CountCalls` 实例里存着调用次数；用闭包实现同样功能得靠 `nonlocal` 或可变容器。它本质上就是一个 GoF 意义上的装饰者对象：持有被装饰者、实现同一接口（可调用）、叠加职责。

In [9]:
import functools


class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)  # 类似 functools.wraps，保留元信息
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"第 {self.count} 次调用 {self.func.__name__}")
        return self.func(*args, **kwargs)


@CountCalls
def ping():
    return "pong"


ping()
ping()
print(f"共调用 {ping.count} 次")

第 1 次调用 ping
第 2 次调用 ping
共调用 2 次


## 十、装饰类本身，以及标准库中的装饰器

`@` 装饰的也可以是类：装饰器接收类、返回类（通常是修改后的同一个类），常见用途有注册（把类登记到某个表里）、批量添加方法等。

标准库大量使用装饰器，常用的有：

| 装饰器 | 作用 |
| --- | --- |
| `@staticmethod` / `@classmethod` | 静态方法 / 类方法（见第 4 篇笔记） |
| `@property` | 把方法包装成属性 |
| `@functools.cache` | 自动缓存函数全部结果 |
| `@functools.lru_cache` | 带容量上限的缓存 |
| `@dataclasses.dataclass` | 自动生成 `__init__`、`__repr__` 等 |
| `@contextlib.contextmanager` | 用生成器实现上下文管理器 |

In [10]:
def add_repr(cls):
    def __repr__(self):
        args = ", ".join(f"{v!r}" for v in vars(self).values())
        return f"{cls.__name__}({args})"

    cls.__repr__ = __repr__  # 给类批量添加方法
    return cls


@add_repr
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


print(Point(1, 2))

Point(1, 2)


In [11]:
import functools


@functools.cache  # 缓存每次调用的结果，递归瞬间完成
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(80))

23416728348467685


## 十一、两种「装饰器」的关系

| | 装饰器设计模式（GoF） | Python 装饰器 |
| --- | --- | --- |
| 包装对象 | 实现了同一接口的对象 | 任意可调用对象（函数、方法、类） |
| 实现方式 | 定义装饰类，组合持有被装饰对象，手动逐层包裹 | `@` 语法 + 高阶函数或实现了 `__call__` 的类，定义时自动包裹 |
| 添加职责 | 在接口方法的前后插入逻辑 | 在 wrapper 的调用前后插入逻辑 |
| 叠加 | 手动嵌套构造 | 多个 `@` 自动叠加，从下往上 |
| 典型场景 | 运行期动态组合行为（流、GUI 组件、咖啡配料） | 日志、计时、缓存、重试、注册、权限校验 |

共同思想：**组合优于继承**、对外接口不变、运行期叠加职责。区别只在成本：GoF 版本每个职责都要一个类、每次使用都要手动嵌套；Python 的 `@` 把「包装」交给解释器自动完成，让装饰器模式便宜到日常随手可用。

## 小练习

实现一个带参数的装饰器 `retry(times)`：调用失败（抛出异常）时自动重试，最多 `times` 次，每次失败打印一条提示；全部失败则抛出最后一次的异常。

提示：三层结构——`retry(times)` 返回 `decorator`，`decorator` 返回 `wrapper`，`wrapper` 里用 `for` + `try/except` 完成重试。

In [12]:
import functools


def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ValueError as e:
                    last_error = e
                    print(f"第 {attempt} 次调用失败：{e}")
            raise last_error

        return wrapper

    return decorator


attempts = 0


@retry(times=3)
def fetch_data():
    global attempts
    attempts += 1
    if attempts < 3:
        raise ValueError("服务暂时不可用")
    return "数据"


print(fetch_data())

第 1 次调用失败：服务暂时不可用
第 2 次调用失败：服务暂时不可用
数据


## 总结

- 装饰器模式：用**组合**（包装）代替继承，在不修改原对象的前提下叠加职责；装饰者与被装饰者实现**同一接口**，装饰者内部**持有**被装饰者。
- Python 函数是一等对象，「接收函数、返回增强版新函数」就是函数版的装饰器模式；`@` 只是把手动包装变成定义时的自动步骤，且**只执行一次**。
- 写装饰器的固定套路：`wrapper(*args, **kwargs)` 转交参数和返回值；用 `functools.wraps` 保留元信息；需要配置时再包一层形成装饰器工厂；需要状态时改用实现了 `__call__` 的类。
- 多个装饰器从下往上装饰、从外往里执行。
- 日志、计时、缓存、重试、注册、权限校验……凡是想在「调用前后」统一插入的逻辑，都适合做成装饰器。